In [0]:
# Inject WHL to bypass Serverless environment dependencies crash
%pip install /Workspace/Users/xh.shi0507@gmail.com/.bundle/enterprise-market-macro-lakehouse/artifacts/.internal/lakehouse-0.1.0-py3-none-any.whl -q
dbutils.library.restartPython()

# cell 0 — 安装 WHL（Serverless）

# 1. Imports

In [0]:
import json
import uuid
from datetime import datetime, timezone

import requests
from pyspark.sql import Row
from pyspark.sql import types as T

# 2. Widget Definitions

In [0]:
# Interface definitions
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_ref")
dbutils.widgets.text("table", "fred_series_raw")

# dbutils.widgets.text("series_ids", "FEDFUNDS,CPIAUCSL")  # List of Series IDs every month
dbutils.widgets.text("series_ids", "DFF,CPIAUCSL")  # List of Series IDs every working day

dbutils.widgets.text("file_type", "json")
dbutils.widgets.text("api_key", "")  # Security: Use secrets in prod
dbutils.widgets.text("start_date", "")  # Optional: YYYY-MM-DD
dbutils.widgets.text("end_date", "")  # Optional: YYYY-MM-DD

# 3. Configuration & Validation

In [0]:
# 1. Parse Parameters
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
TABLE = dbutils.widgets.get("table").strip()
FILE_TYPE = dbutils.widgets.get("file_type").strip()
# API_KEY = dbutils.widgets.get("api_key").strip()
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

# Parse comma-separated list
SERIES_IDS = [s.strip() for s in dbutils.widgets.get("series_ids").split(",") if s.strip()]

TARGET_TBL = f"{CATALOG}.{SCHEMA}.{TABLE}"

# 2. Validations
if not SERIES_IDS:
    raise ValueError("series_ids is empty. Please provide at least one series ID (e.g. DFF).")

"""
if not API_KEY:
    raise ValueError(
        "FRED api_key is required. Please provide it via the widget.\n"
        "Recommendation: Use dbutils.secrets.get() in production."
    )
"""

# --- FRED API KEY: prefer secrets, fallback to widget (dev) ---
API_KEY = ""

# 1️⃣ Try secret first (production / job run)
try:
    API_KEY = dbutils.secrets.get("enterprise", "fred_api_key")
except Exception:
    API_KEY = ""

# 2️⃣ Fallback to widget (manual testing)
if not API_KEY:
    try:
        API_KEY = dbutils.widgets.get("api_key").strip()
    except Exception:
        API_KEY = ""

if not API_KEY:
    raise ValueError(
        "FRED api_key is required. Provide via secret scope enterprise/fred_api_key or widget api_key."
    )


# 3. Run Metadata
run_id = str(uuid.uuid4())
ingestion_ts = datetime.now(timezone.utc)

print("[INFO] Configuration Ready:")
print(f"       Target: {TARGET_TBL}")
print(f"       Series Count: {len(SERIES_IDS)} -> {SERIES_IDS}")
print(f"       Run ID: {run_id}")

# 4. DDL / Table Setup

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TBL} (
  series_id     STRING,
  raw_json      STRING,
  run_id        STRING,
  ingestion_ts  TIMESTAMP
)
USING DELTA
COMMENT 'Raw ingestion table for FRED economic data. Contains full JSON observations.'
""")

# 5. Data Extraction

In [0]:
print(f"[INFO] Starting extraction for {len(SERIES_IDS)} series...")
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

rows = []
failed_series = []

for sid in SERIES_IDS:
    try:
        params = {"series_id": sid, "api_key": API_KEY, "file_type": FILE_TYPE}
        if START_DATE:
            params["observation_start"] = START_DATE
        if END_DATE:
            params["observation_end"] = END_DATE

        # Request data
        r = requests.get(BASE_URL, params=params, timeout=30)
        r.raise_for_status()

        # Compress JSON to save storage space
        if FILE_TYPE == "json":
            raw_payload = json.dumps(r.json(), separators=(",", ":"))
        else:
            raw_payload = r.text

        rows.append(
            Row(series_id=sid, raw_json=raw_payload, run_id=run_id, ingestion_ts=ingestion_ts)
        )
        print(f"   -> [OK] {sid}")

    except Exception as e:
        print(f"   -> [ERROR] Failed to fetch {sid}: {e}")
        failed_series.append(sid)

print(f"[INFO] Extraction finished. Success: {len(rows)}, Failed: {len(failed_series)}")
if failed_series:
    print(f"[WARN] The following series failed: {failed_series}")

# 6. Data Writing

In [0]:
if rows:
    schema = T.StructType(
        [
            T.StructField("series_id", T.StringType(), False),
            T.StructField("raw_json", T.StringType(), False),
            T.StructField("run_id", T.StringType(), False),
            T.StructField("ingestion_ts", T.TimestampType(), False),
        ]
    )

    df_bronze = spark.createDataFrame(rows, schema=schema)

    print(f"[INFO] Writing FRED rows to {TARGET_TBL}...")
    (df_bronze.write.mode("append").format("delta").saveAsTable(TARGET_TBL))
    print("[INFO] Write completed.")
else:
    print("[WARN] No data to write (rows list is empty).")

# 7. Verification